### #1. Parámetros y Generación de Datos
Generamos un conjunto de datos sintético aleatorio que simula lecturas de sensores tanto normales como anómalas.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, make_scorer
from sklearn.inspection import permutation_importance

# ==========================================
# 1. Parámetros y Generación de Datos
# ==========================================
N = 1000  # Número total de datos a generar

# Generamos con los nombres originales de los sensores del sistema externo
RANGES_SANA = {
    'Air_sensor_Temperature[C]': (19.0, 29.0),
    'Air_sensor_Humidity[%RH]': (45.0, 75.0),
    '7in1_Nitrogen[mg/kg]': (15, 45),
    '7in1_Phosphorus[mg/kg]': (18, 55),
    '7in1_Potasium[mg/kg]': (45, 245),
    '7in1_Moisture[%RH]': (20.0, 43.0),
    '7in1_S_Temperature[C]': (21.0, 29.0),
    '7in1_Conductivity[uS/cm]': (200, 400),
    '7in1_Ph[pH]': (6, 6.5)
}

# Ajustamos las anomalías para que estén más cerca e incluso se superpongan ligeramente con los sanos
RANGES_ANOMALOS_ALTA = {
    'Air_sensor_Temperature[C]': (27.0, 34.0),
    'Air_sensor_Humidity[%RH]': (68.0, 85.0),
    '7in1_Nitrogen[mg/kg]': (40, 65),
    '7in1_Phosphorus[mg/kg]': (48, 70),
    '7in1_Potasium[mg/kg]': (200, 280),
    '7in1_Moisture[%RH]': (38.0, 55.0),
    '7in1_S_Temperature[C]': (27.0, 34.0),
    '7in1_Conductivity[uS/cm]': (350, 500),
    '7in1_Ph[pH]': (6.3, 7.0)
}

RANGES_ANOMALOS_BAJA = {
    'Air_sensor_Temperature[C]': (14.0, 21.0),
    'Air_sensor_Humidity[%RH]': (35.0, 50.0),
    '7in1_Nitrogen[mg/kg]': (8, 20),
    '7in1_Phosphorus[mg/kg]': (10, 22),
    '7in1_Potasium[mg/kg]': (20, 60),
    '7in1_Moisture[%RH]': (10.0, 24.0),
    '7in1_S_Temperature[C]': (15.0, 23.0),
    '7in1_Conductivity[uS/cm]': (100, 220),
    '7in1_Ph[pH]': (5.0, 6.1)
}

# --- ANOMALÍAS EXCLUSIVAS DE SUELO ---
RANGES_SUELO_SECO = {'7in1_Moisture[%RH]': (10.0, 24.0), '7in1_S_Temperature[C]': (27.0, 34.0)}
RANGES_SUELO_INUNDADO = {'7in1_Moisture[%RH]': (38.0, 55.0), '7in1_S_Temperature[C]': (15.0, 23.0)}

# Anomalías mixtas
RANGES_FRIO_HUMEDO = {'Air_sensor_Temperature[C]': (14.0, 21.0), 'Air_sensor_Humidity[%RH]': (68.0, 85.0), '7in1_S_Temperature[C]': (15.0, 23.0), '7in1_Moisture[%RH]': (38.0, 55.0)}
RANGES_CALIENTE_SECO = {'Air_sensor_Temperature[C]': (27.0, 34.0), 'Air_sensor_Humidity[%RH]': (35.0, 50.0), '7in1_S_Temperature[C]': (27.0, 34.0), '7in1_Moisture[%RH]': (10.0, 24.0)}
RANGES_EXCESO_FERTILIZANTE = {'7in1_Nitrogen[mg/kg]': (40, 75), '7in1_Phosphorus[mg/kg]': (48, 80), '7in1_Potasium[mg/kg]': (200, 300), '7in1_Conductivity[uS/cm]': (350, 600), '7in1_Ph[pH]': (5.0, 6.1)}
RANGES_LAVADO_NUTRIENTES = {'7in1_Nitrogen[mg/kg]': (8, 18), '7in1_Phosphorus[mg/kg]': (10, 20), '7in1_Potasium[mg/kg]': (20, 50), '7in1_Conductivity[uS/cm]': (100, 220), '7in1_Ph[pH]': (6.3, 7.2)}

np.random.seed(42)

def generar_muestra(n, rangos_base, rangos_especificos=None):
    data = {}
    for col in RANGES_SANA.keys():
        rango = rangos_especificos.get(col, rangos_base[col]) if rangos_especificos else rangos_base[col]
        data[col] = np.random.uniform(rango[0], rango[1], n)
    return pd.DataFrame(data)

# Distribución de la generación de datos
n_sanos = int(N * 0.40)
n_alta = int(N * 0.05)
n_baja = int(N * 0.05)
n_suelo = int(N * 0.15)
n_frio = int(N * 0.05)
n_caliente = int(N * 0.05)
n_exceso = int(N * 0.125)
n_lavado = N - (n_sanos + n_alta + n_baja + n_suelo + n_frio + n_caliente + n_exceso)

# Generar subconjuntos
df_sanos = generar_muestra(n_sanos, RANGES_SANA)
df_alta = generar_muestra(n_alta, RANGES_ANOMALOS_ALTA)
df_baja = generar_muestra(n_baja, RANGES_ANOMALOS_BAJA)

df_suelo_seco = generar_muestra(n_suelo // 2, RANGES_SANA, RANGES_SUELO_SECO)
df_suelo_inundado = generar_muestra(n_suelo - (n_suelo // 2), RANGES_SANA, RANGES_SUELO_INUNDADO)

df_frio = generar_muestra(n_frio, RANGES_SANA, RANGES_FRIO_HUMEDO)
df_caliente = generar_muestra(n_caliente, RANGES_SANA, RANGES_CALIENTE_SECO)
df_exceso = generar_muestra(n_exceso, RANGES_SANA, RANGES_EXCESO_FERTILIZANTE)
df_lavado = generar_muestra(n_lavado, RANGES_SANA, RANGES_LAVADO_NUTRIENTES)

# Concatenar y mezclar
df = pd.concat([df_sanos, df_alta, df_baja, df_suelo_seco, df_suelo_inundado, df_frio, df_caliente, df_exceso, df_lavado], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Renombrar columnas para usar "alias" y coincidir exactamente con la app
df = df.rename(columns={
    '7in1_Nitrogen[mg/kg]': 'nitrogeno',
    '7in1_Phosphorus[mg/kg]': 'fosforo',
    '7in1_Potasium[mg/kg]': 'potasio',
    '7in1_Ph[pH]': 'ph',
    '7in1_Conductivity[uS/cm]': 'conductividad',
    '7in1_S_Temperature[C]': 'temperatura_suelo',
    '7in1_Moisture[%RH]': 'humedad_suelo',
    'Air_sensor_Temperature[C]': 'temperatura',
    'Air_sensor_Humidity[%RH]': 'humedad'
})

# Definición de variables con los nuevos nombres (alias)
col_n = 'nitrogeno'
col_p = 'fosforo'
col_k = 'potasio'
col_ph = 'ph'
col_ec = 'conductividad'
col_temp_suelo = 'temperatura_suelo'
col_hum_suelo = 'humedad_suelo'
col_temp = 'temperatura'
col_hum = 'humedad'
col_tratamiento = 'Tratamiento'

features = list(df.columns)
display(df.head())


,temperatura,humedad,nitrogeno,fosforo,potasio,humedad_suelo,temperatura_suelo,conductividad,ph
0,24.771171,67.356442,41.378387,25.025387,102.793705,12.060286,33.285687,297.957174,6.462321
1,28.329543,46.139540,18.442259,32.193534,222.654439,10.688077,31.111589,270.341964,6.001915
2,28.732241,36.322134,23.065846,40.340332,82.699971,16.355424,32.294492,358.026275,6.085543
3,17.187764,74.907648,27.010699,47.736658,125.703998,54.600324,16.238288,299.303084,6.060241
4,28.001008,76.521686,57.901249,64.168369,236.634320,48.401127,28.903990,408.934266,6.574337


### #1.5 Agregar Temporalidad y Ajustar Temperaturas
Simulamos el paso del tiempo asignando horas y meses, ajustando la temperatura para que sea menor en la madrugada/invierno, y extraemos las características cíclicas.

In [2]:
# Generar horas y meses aleatorios para nuestro dataset sintético
df['mes'] = np.random.randint(1, 13, size=len(df))
df['hora'] = np.random.randint(0, 24, size=len(df))

# Simular la caída de temperatura:
# factor_hora: disminuye de madrugada (aprox a las 3-4 AM) y sube en la tarde.
factor_hora = -3.0 * np.cos((df['hora'] - 3) * (2 * np.pi / 24))

# factor_mes: simulamos un invierno (ej. Hemisferio Norte, enero más frío)
# Si necesitas hemisferio sur, puedes cambiar el desfase.
factor_mes = -5.0 * np.cos((df['mes'] - 1) * (2 * np.pi / 12))

# Aplicamos la variación de la temperatura ambiente y del suelo (el suelo varía menos)
df['temperatura'] += factor_hora + factor_mes
df['temperatura_suelo'] += (factor_hora + factor_mes) * 0.4

# Generar variables cíclicas para que el Random Forest pueda entender la continuidad del tiempo
df['hora_sin'] = np.sin(2 * np.pi * df['hora'] / 24)
df['hora_cos'] = np.cos(2 * np.pi * df['hora'] / 24)
df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)

# Características ORIGINALES (13D). Ya no se entregan directamente a los modelos:
# son la entrada de la reducción PCA (sección #3), que produce las 3 componentes
# con las que se entrenan los modelos.
# ORDEN AJUSTADO para que coincida exactamente con el archivo de la aplicación (FastAPI)
features = [
    'nitrogeno', 'fosforo', 'potasio', 'ph', 'conductividad',
    'humedad_suelo', 'temperatura_suelo', 'temperatura', 'humedad',
    'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos'
]

display(df[['mes', 'hora', 'temperatura', 'temperatura_suelo', 'hora_cos', 'mes_cos']].head())

,mes,hora,temperatura,temperatura_suelo,hora_cos,mes_cos
0,5,13,29.869247,35.324917,-0.965926,-8.660254e-01
1,2,9,23.999416,29.379538,-0.707107,5.000000e-01
2,12,12,26.523434,31.410970,-1.000000,1.000000e+00
3,1,11,13.687764,14.838288,-0.965926,8.660254e-01
4,9,0,28.379688,29.055462,1.000000,-1.836970e-16


### #2. Reglas Deterministas (Ground Truth)
Establecemos las reglas lógicas que definen si una planta está sana o qué tipo de anomalía presenta.

In [3]:
# ==========================================
# 2. Reglas Deterministas (Ground Truth)
# ==========================================
# Rangos correspondientes a una planta sana
RANGES_SANA = {
    "temperatura": (18.0, 30.0),
    "humedad": (44.0, 75.0),
    "nitrogeno": (10, 50),
    "fosforo": (15, 60),
    "potasio": (40, 250),
    "humedad_suelo": (15.0, 45.0),
    "temperatura_suelo": (20.0, 30.0),
    "conductividad": (100, 500),
    "ph": (5.8, 6.8),
}

def diagnosticar_por_reglas(row):
    if not (RANGES_SANA["temperatura"][0] <= row['temperatura'] <= RANGES_SANA["temperatura"][1]) or \
       not (RANGES_SANA["humedad"][0] <= row['humedad'] <= RANGES_SANA["humedad"][1]):
        return 'Anomalía de Ambiente'

    if not (RANGES_SANA["humedad_suelo"][0] <= row['humedad_suelo'] <= RANGES_SANA["humedad_suelo"][1]) or \
       not (RANGES_SANA["temperatura_suelo"][0] <= row['temperatura_suelo'] <= RANGES_SANA["temperatura_suelo"][1]):
        return 'Anomalía de Suelo'

    if not (RANGES_SANA["nitrogeno"][0] <= row['nitrogeno'] <= RANGES_SANA["nitrogeno"][1]) or \
       not (RANGES_SANA["fosforo"][0] <= row['fosforo'] <= RANGES_SANA["fosforo"][1]) or \
       not (RANGES_SANA["potasio"][0] <= row['potasio'] <= RANGES_SANA["potasio"][1]) or \
       not (RANGES_SANA["conductividad"][0] <= row['conductividad'] <= RANGES_SANA["conductividad"][1]) or \
       not (RANGES_SANA["ph"][0] <= row['ph'] <= RANGES_SANA["ph"][1]):
        return 'Anomalía de Nutrientes'

    return 'Sin anomalías (Sana)'

# Aplicar las reglas para generar la etiqueta real de entrenamiento
df['Etiqueta_Real'] = df.apply(diagnosticar_por_reglas, axis=1)
print(df['Etiqueta_Real'].value_counts())

Etiqueta_Real
Anomalía de Ambiente      379
Sin anomalías (Sana)      300
Anomalía de Nutrientes    179
Anomalía de Suelo         142
Name: count, dtype: int64


### #3. Reducción Dimensional con PCA (13D → 3D)
Reducimos las 13 características a 3 componentes principales (una por grupo: nutrientes, suelo y ambiente). Estas 3 componentes se usan tanto para **visualizar** como para **entrenar y alimentar a los modelos**.

Los escaladores y los PCA se ajustan una sola vez y se reutilizan en el entrenamiento, en la prueba interactiva y en la exportación, de modo que la aplicación aplique exactamente la misma transformación.

In [4]:
# ==========================================
# 3. PCA: Reducción Dimensional (13D -> 3D)
# ==========================================
# Definición de los 3 grupos de variables
grupo_nutrientes = ['nitrogeno', 'fosforo', 'potasio', 'ph', 'conductividad']
grupo_suelo = ['humedad_suelo', 'temperatura_suelo']
grupo_ambiente = ['temperatura', 'humedad', 'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos']

# Estandarización (se ajusta una sola vez; estos mismos objetos se exportan para la app)
scaler_nut = StandardScaler().fit(df[grupo_nutrientes])
scaler_suelo = StandardScaler().fit(df[grupo_suelo])
scaler_amb = StandardScaler().fit(df[grupo_ambiente])

# Extracción de 1 componente principal por grupo
pca_nut = PCA(n_components=1, random_state=42).fit(scaler_nut.transform(df[grupo_nutrientes]))
pca_suelo = PCA(n_components=1, random_state=42).fit(scaler_suelo.transform(df[grupo_suelo]))
pca_amb = PCA(n_components=1, random_state=42).fit(scaler_amb.transform(df[grupo_ambiente]))

# Las 3 componentes son las ÚNICAS entradas que recibirán los modelos
features_pca = ['Comp_Nutrientes', 'Comp_Suelo', 'Comp_Ambiente']

def proyectar_pca(datos):
    """Convierte las 13 características originales (DataFrame) en las 3 componentes PCA."""
    return pd.DataFrame({
        'Comp_Nutrientes': pca_nut.transform(scaler_nut.transform(datos[grupo_nutrientes])).ravel(),
        'Comp_Suelo': pca_suelo.transform(scaler_suelo.transform(datos[grupo_suelo])).ravel(),
        'Comp_Ambiente': pca_amb.transform(scaler_amb.transform(datos[grupo_ambiente])).ravel(),
    }, index=datos.index)[features_pca]

componentes = proyectar_pca(df)
for col in features_pca:
    df[col] = componentes[col]

# Diagnóstico: cuánta información conserva cada componente y qué variables pesan en ella
print("--- VARIANZA EXPLICADA Y PESOS (loadings) DE CADA COMPONENTE ---")
for nombre, pca, grupo in [('Nutrientes', pca_nut, grupo_nutrientes),
                           ('Suelo', pca_suelo, grupo_suelo),
                           ('Ambiente', pca_amb, grupo_ambiente)]:
    print(f"\nComp_{nombre}: {pca.explained_variance_ratio_[0] * 100:.1f}% de la varianza del grupo")
    print(pd.Series(pca.components_[0], index=grupo).round(3).to_string())

--- VARIANZA EXPLICADA Y PESOS (loadings) DE CADA COMPONENTE ---

Comp_Nutrientes: 66.4% de la varianza del grupo
nitrogeno        0.477
fosforo          0.478
potasio          0.457
ph              -0.318
conductividad    0.483

Comp_Suelo: 64.1% de la varianza del grupo
humedad_suelo        0.707
temperatura_suelo   -0.707

Comp_Ambiente: 29.5% de la varianza del grupo
temperatura    0.698
humedad        0.023
hora_sin      -0.270
hora_cos      -0.307
mes_sin       -0.271
mes_cos       -0.520


### #4. Entrenamiento de Modelos e Importancia de Variables
Entrenamos un clasificador multiclase (Random Forest) y un detector de anomalías no supervisado (Isolation Forest). **Ambos reciben únicamente las 3 componentes PCA** (`Comp_Nutrientes`, `Comp_Suelo`, `Comp_Ambiente`) en lugar de las 13 características originales. Se analiza su rendimiento y la relevancia de cada componente.

In [5]:
# ==========================================
# 4. Entrenamiento de Modelos (entrada: 3 componentes PCA)
# ==========================================
# 4.1 Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, oob_score=True)
rf.fit(df[features_pca], df['Etiqueta_Real'])
df['Pred_RF'] = rf.predict(df[features_pca])

# 4.2 Isolation Forest
iforest = IsolationForest(n_estimators=100, contamination='auto', random_state=42)
df['Pred_IF_raw'] = iforest.fit_predict(df[features_pca])
df['Pred_IF_Label'] = df['Pred_IF_raw'].map({1: 'Sin anomalías (Sana)', -1: 'Anomalía (General)'})

# ==========================================
# 5. Cálculo de Confiabilidad y Pesos
# ==========================================
acc_rf = accuracy_score(df['Etiqueta_Real'], df['Pred_RF'])
# Un Random Forest suele acertar ~100% sobre sus propios datos de entrenamiento; la estimación
# OOB (out-of-bag: cada muestra se evalúa solo con los árboles que no la vieron) es más realista.
acc_rf_oob = rf.oob_score_
pesos_rf = pd.DataFrame({'Variable': features_pca, 'Peso_RF': rf.feature_importances_}).sort_values(by='Peso_RF', ascending=False)

y_true_binary = df['Etiqueta_Real'].apply(lambda x: 1 if x == 'Sin anomalías (Sana)' else -1)
acc_if = accuracy_score(y_true_binary, df['Pred_IF_raw'])

pi_if = permutation_importance(iforest, df[features_pca], y_true_binary, scoring=make_scorer(accuracy_score), random_state=42)
pesos_if = pd.DataFrame({'Variable': features_pca, 'Peso_IF': pi_if.importances_mean}).sort_values(by='Peso_IF', ascending=False)

# Reporte
print(f"--- RESULTADOS DEL ENTRENAMIENTO (N={N}, entrada: {len(features_pca)} componentes PCA) ---")
print(f"Confiabilidad Random Forest (sobre datos de entrenamiento): {acc_rf * 100:.2f}%")
print(f"Confiabilidad Random Forest (estimación OOB, datos no vistos): {acc_rf_oob * 100:.2f}%")
print(f"Confiabilidad Isolation Forest (Binaria): {acc_if * 100:.2f}%\n")

print("--- PESOS DE VARIABLES: RANDOM FOREST ---")
print(pesos_rf.to_string(index=False))
print("\n--- PESOS DE VARIABLES: ISOLATION FOREST (Permutation Importance) ---")
print(pesos_if.to_string(index=False))

--- RESULTADOS DEL ENTRENAMIENTO (N=1000, entrada: 3 componentes PCA) ---
Confiabilidad Random Forest (sobre datos de entrenamiento): 100.00%
Confiabilidad Random Forest (estimación OOB, datos no vistos): 74.70%
Confiabilidad Isolation Forest (Binaria): 58.50%

--- PESOS DE VARIABLES: RANDOM FOREST ---
       Variable  Peso_RF
Comp_Nutrientes 0.381877
  Comp_Ambiente 0.346833
     Comp_Suelo 0.271290

--- PESOS DE VARIABLES: ISOLATION FOREST (Permutation Importance) ---
       Variable  Peso_IF
Comp_Nutrientes   0.0726
     Comp_Suelo   0.0274
  Comp_Ambiente   0.0068


### #5. Visualización Interactiva 3D
Generamos gráficos interactivos en 3D utilizando Plotly para comparar visualmente los diagnósticos de ambos modelos. Los ejes son las mismas 3 componentes PCA que reciben los modelos; al pasar el cursor se muestran las 13 características originales.

In [6]:
# ==========================================
# 6. Visualización 3D
# ==========================================
color_map_rf = {
    'Sin anomalías (Sana)': '#2E7D32',
    'Anomalía de Nutrientes': '#C62828',
    'Anomalía de Suelo': '#F57F17',
    'Anomalía de Ambiente': '#1565C0'
}

color_map_if = {
    'Sin anomalías (Sana)': '#2E7D32',
    'Anomalía (General)': '#8E24AA'
}

# Gráfico 1: Random Forest
fig_rf = px.scatter_3d(
    df, x='Comp_Nutrientes', y='Comp_Suelo', z='Comp_Ambiente',
    color='Pred_RF',
    color_discrete_map=color_map_rf,
    title=f"1. Clasificación Aprendida por Random Forest (Confiabilidad: {acc_rf*100:.1f}%)",
    hover_data=features
)
fig_rf.update_traces(marker=dict(size=6, opacity=0.8))
fig_rf.show()

# Gráfico 2: Isolation Forest
fig_if = px.scatter_3d(
    df, x='Comp_Nutrientes', y='Comp_Suelo', z='Comp_Ambiente',
    color='Pred_IF_Label',
    color_discrete_map=color_map_if,
    title=f"2. Asignación de Anomalías por Isolation Forest (Confiabilidad Binaria: {acc_if*100:.1f}%)",
    hover_data=features
)
fig_if.update_traces(marker=dict(size=6, opacity=0.8))
fig_if.show()

### #7. Prueba Interactiva de Predicción
Utiliza los siguientes controles para introducir valores manualmente y verificar la respuesta de los modelos en tiempo real.

In [7]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import pandas as pd

# Crear widgets de entrada para cada variable
w_temp = widgets.FloatText(value=25.0, description='Temp Aire (°C):', style={'description_width': 'initial'})
w_hum = widgets.FloatText(value=60.0, description='Humedad Aire (%):', style={'description_width': 'initial'})
w_n = widgets.FloatText(value=30.0, description='Nitrógeno (mg/kg):', style={'description_width': 'initial'})
w_p = widgets.FloatText(value=30.0, description='Fósforo (mg/kg):', style={'description_width': 'initial'})
w_k = widgets.FloatText(value=100.0, description='Potasio (mg/kg):', style={'description_width': 'initial'})
w_hum_suelo = widgets.FloatText(value=30.0, description='Humedad Suelo (%):', style={'description_width': 'initial'})
w_temp_suelo = widgets.FloatText(value=22.0, description='Temp Suelo (°C):', style={'description_width': 'initial'})
w_ec = widgets.FloatText(value=200.0, description='Conductividad (uS/cm):', style={'description_width': 'initial'})
w_ph = widgets.FloatText(value=6.0, description='pH:', style={'description_width': 'initial'})

# NUEVO: hora y mes. El grupo Ambiente del PCA incluye las variables cíclicas (seno/coseno),
# así que se necesitan para poder calcular la componente PCA correspondiente.
w_hora = widgets.BoundedIntText(value=12, min=0, max=23, description='Hora (0-23):', style={'description_width': 'initial'})
w_mes = widgets.BoundedIntText(value=6, min=1, max=12, description='Mes (1-12):', style={'description_width': 'initial'})

btn_predict = widgets.Button(description="Predecir", button_style='success')
output = widgets.Output()

def on_predict_clicked(b):
    with output:
        output.clear_output()
        hora, mes = w_hora.value, w_mes.value

        # Crear DataFrame con los valores de entrada y los alias dinámicos
        input_data = pd.DataFrame([{
            col_n: w_n.value,
            col_p: w_p.value,
            col_k: w_k.value,
            col_ph: w_ph.value,
            col_ec: w_ec.value,
            col_hum_suelo: w_hum_suelo.value,
            col_temp_suelo: w_temp_suelo.value,
            col_temp: w_temp.value,
            col_hum: w_hum.value,
            'hora_sin': np.sin(2 * np.pi * hora / 24),
            'hora_cos': np.cos(2 * np.pi * hora / 24),
            'mes_sin': np.sin(2 * np.pi * mes / 12),
            'mes_cos': np.cos(2 * np.pi * mes / 12),
        }])

        # Características originales (13D) en el mismo orden usado en el entrenamiento
        input_data = input_data[features]

        # Reducción 13D -> 3D con los mismos escaladores y PCA ajustados en el entrenamiento
        input_pca = proyectar_pca(input_data)

        # Predecir: ambos modelos fueron entrenados con df[features_pca]
        pred_if = iforest.predict(input_pca)[0]
        pred_rf = rf.predict(input_pca)[0]

        if_label = "Sin anomalías (1)" if pred_if == 1 else "Anomalía detectada (-1)"

        print("=== Resultados de la Predicción ===")
        print("Componentes PCA -> " + " | ".join(f"{c}: {input_pca[c].iloc[0]:.3f}" for c in features_pca))
        print(f"Isolation Forest (Detección de outliers): {if_label}")
        print(f"Random Forest (Clasificación por reglas): {pred_rf}")

btn_predict.on_click(on_predict_clicked)

form = widgets.VBox([
    widgets.HTML("<b>Ingresa los valores controlados para probar el modelo:</b>"),
    w_temp, w_hum, w_n, w_p, w_k, w_hum_suelo, w_temp_suelo, w_ec, w_ph,
    w_hora, w_mes,
    btn_predict,
    output
])

display(form)

In [8]:
import joblib

# ==========================================
# 8. Exportación de Modelos a .joblib
# ==========================================
# Los escaladores y PCA ya fueron ajustados en la sección #3 y son EXACTAMENTE los que se usaron
# para entrenar los modelos (no se vuelven a ajustar aquí). Se exportan junto a cada modelo para
# que la aplicación aplique la misma transformación 13D -> 3D antes de predecir.
preprocesadores = {
    'scaler_nut': scaler_nut,
    'scaler_suelo': scaler_suelo,
    'scaler_amb': scaler_amb,
    'pca_nut': pca_nut,
    'pca_suelo': pca_suelo,
    'pca_amb': pca_amb,
    'features': features,            # 13 características originales (entrada de la app)
    'features_pca': features_pca,    # 3 componentes (entrada real de los modelos)
    'grupo_nutrientes': grupo_nutrientes,
    'grupo_suelo': grupo_suelo,
    'grupo_ambiente': grupo_ambiente
}

# 1. Exportar Isolation Forest junto con los preprocesadores requeridos
iforest_pack = {**preprocesadores, 'iforest': iforest}
nombre_iforest = 'modelo_isolation_forest_pca.joblib'
joblib.dump(iforest_pack, nombre_iforest)
print(f"✅ Isolation Forest exportado correctamente en '{nombre_iforest}'.")

# 2. Exportar Random Forest junto con los preprocesadores requeridos (la app lo carga con este nombre)
rf_pack = {**preprocesadores, 'rf': rf}
nombre_rf = 'rf_model_pca.joblib'
joblib.dump(rf_pack, nombre_rf)
print(f"✅ Random Forest exportado correctamente en '{nombre_rf}'.")

# Opcional: Descargar los archivos localmente a tu PC desde Colab
try:
    from google.colab import files
    #files.download(nombre_iforest)
    files.download(nombre_rf) #descomentar para descargar random forest
    print("⌛ Iniciando descarga automática de los modelos...")
except Exception as e:
    print("Nota: Si estás usando un navegador con bloqueador de ventanas emergentes, puedes descargar los archivos manualmente desde el panel izquierdo de archivos.")

✅ Isolation Forest exportado correctamente en 'modelo_isolation_forest_pca.joblib'.
✅ Random Forest exportado correctamente en 'rf_model_pca.joblib'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⌛ Iniciando descarga automática de los modelos...
